# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
import gradio as gr
import requests
import json
import ollama

# ==============================
# System Prompt (Expert Mode)
# ==============================

SYSTEM_PROMPT = """
You are a senior software engineer and technical instructor.
Explain answers clearly, step-by-step.
Use examples where possible.
Keep explanations structured and beginner-friendly.
"""

# ==============================
# Simple Tool (Bonus Requirement)
# ==============================

def detect_if_code(question):
    """
    Simple tool that detects if input looks like code
    """
    keywords = ["def ", "class ", "import ", "{", "}", "();"]
    return any(k in question for k in keywords)


# ==============================
# Streaming Response Function
# ==============================

def ask_llm(question, model):

    full_prompt = f"{SYSTEM_PROMPT}\n\nUser Question:\n{question}"

    stream = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ],
        stream=True
    )

    response_text = ""

    for chunk in stream:
        content = chunk['message']['content']
        response_text += content
        yield response_text


# ==============================
# Gradio UI
# ==============================

with gr.Blocks() as demo:
    gr.Markdown("# 🧠 Technical Question Answerer Pro")

    model_selector = gr.Dropdown(
        choices=["llama3.2", "mistral"],
        value="llama3.2",
        label="Choose Model"
    )

    question_input = gr.Textbox(
        lines=6,
        placeholder="Ask a technical question...",
        label="Your Question"
    )

    output = gr.Textbox(
        lines=15,
        label="Response",
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        ask_llm,
        inputs=[question_input, model_selector],
        outputs=output
    )

demo.launch()